# P1 · EDA в разрезе user–item (датасет `tgbn-genre`)

**Фаза:** P1 из [`framework.md`](../framework.md). **Вопрос:** жива ли идея плотного per-(user, item) состояния?

**Gate:** подтвердить, что в данных есть *эксплуатируемая* персональная (per-pair) структура — персональная история юзера предсказывает его интересы лучше, чем глобальная популярность. Если нет — плотная матрица `N_user × N_item` бесполезна.

Конвенции: Polars, Plotly, текст на русском. Kernel — conda-env `tgb`.

In [1]:
import polars as pl
import plotly.express as px

DS = "/Users/aleksandrpanysev/miniconda3/envs/tgb/lib/python3.13/site-packages/tgb/datasets"
LABELS = f"{DS}/tgbn_genre/tgbn-genre_node_labels.csv"

# Метки tgbn — это дневное распределение интереса юзера по жанрам (ts, user_id, genre, weight)
lab = pl.read_csv(LABELS)
N_user, N_item, N_ts = lab["user_id"].n_unique(), lab["genre"].n_unique(), lab["ts"].n_unique()
print(f"строк меток: {lab.height:,} | юзеров: {N_user} | жанров (item): {N_item} | дней (ts): {N_ts}")
lab.head()

строк меток: 2,741,935 | юзеров: 974 | жанров (item): 513 | дней (ts): 1579


ts,user_id,genre,weight
i64,str,str,f64
1108443600,"""user_000054""","""chillout""",0.015835
1108443600,"""user_000054""","""female vocalist""",0.01533
1108443600,"""user_000054""","""downtempo""",0.008128
1108443600,"""user_000054""","""electronic""",0.072162
1108443600,"""user_000054""","""reggae""",0.021465


## 1. Разреженность распределения интереса (user × item)

Сколько жанров реально активно у юзера в каждый день. Малый носитель → персональное распределение разрежено, и плотная матрица в основном нулевая (но дёшево хранимая).

In [2]:
# носитель (число активных жанров) и сумма весов на (user, день)
per_uts = lab.group_by(["ts", "user_id"]).agg(
    pl.col("weight").sum().alias("wsum"),
    pl.len().alias("support"),
)
print("сумма весов на (user,день): mean={:.3f} std={:.3f}  → метки {}нормированы".format(
    per_uts["wsum"].mean(), per_uts["wsum"].std(),
    "" if abs(per_uts["wsum"].mean() - 1) < 0.05 else "НЕ "))
print("носитель (актив. жанров) на (user,день): mean={:.1f} median={:.0f} max={}".format(
    per_uts["support"].mean(), per_uts["support"].median(), per_uts["support"].max()))
print("плотность матрицы интереса: {:.1f}% (из {} жанров)".format(
    per_uts["support"].mean() / N_item * 100, N_item))

fig = px.histogram(per_uts.to_pandas(), x="support", nbins=60,
                   title="P1 · genre: размер носителя интереса на (user, день)")
fig.update_layout(xaxis_title="число активных жанров", yaxis_title="кол-во (user, день)")
fig.show()

сумма весов на (user,день): mean=1.000 std=0.000  → метки нормированы
носитель (актив. жанров) на (user,день): mean=10.7 median=8 max=152
плотность матрицы интереса: 2.1% (из 513 жанров)


## 2. Концентрация глобального prior'а

Если интерес сосредоточен на немногих жанрах глобально, то сильный неперсональный baseline (популярность) уже объясняет много, и персональной компоненте нужно бить именно его.

In [3]:
# глобальная популярность жанра = суммарная масса интереса по всем (user, день)
pop = (lab.group_by("genre").agg(pl.col("weight").sum().alias("mass"))
          .sort("mass", descending=True)
          .with_columns((pl.col("mass") / pl.col("mass").sum()).alias("share")))
pop = pop.with_columns(pl.col("share").cum_sum().alias("cum_share")).with_row_index("rank")

for k in [10, 20, 50, 100]:
    print(f"top-{k:>3} жанров покрывают {pop['cum_share'][k-1]*100:5.1f}% всей массы интереса")

fig = px.line(pop.to_pandas(), x="rank", y="cum_share",
              title="P1 · genre: накопленная доля массы по популярности жанров")
fig.update_layout(xaxis_title="ранг жанра (по популярности)", yaxis_title="накопленная доля массы")
fig.show()

top- 10 жанров покрывают  41.9% всей массы интереса
top- 20 жанров покрывают  53.9% всей массы интереса
top- 50 жанров покрывают  74.3% всей массы интереса
top-100 жанров покрывают  87.4% всей массы интереса


## 3. Persistence: «перенесённая» масса интереса (repeat-mass)

Для каждого (user, день) — какая доля сегодняшней массы интереса приходится на жанры, с которыми юзер взаимодействовал в свой **предыдущий активный день**. Высокое значение → поведение повторяющееся → per-(user, item) память — правильный индуктивный bias (и объясняет, почему persistent-forecast так силён).

In [4]:
# нормируем веса в распределение p на (user, день)
norm = (lab.join(per_uts.select(["ts", "user_id", "wsum"]), on=["ts", "user_id"])
           .with_columns((pl.col("weight") / pl.col("wsum")).alias("p")))

# для каждого юзера — предыдущий его активный день
uts = norm.select(["user_id", "ts"]).unique().sort(["user_id", "ts"])
uts = uts.with_columns(pl.col("ts").shift(1).over("user_id").alias("prev_ts"))
pairs = uts.drop_nulls("prev_ts")  # (user, ts, prev_ts)

cur = norm.select(["user_id", "ts", "genre", "p"])
prev_supp = (norm.select(["user_id", pl.col("ts").alias("prev_ts"), "genre"])
                 .unique().with_columns(pl.lit(1).alias("in_prev")))

carried = (pairs.join(cur, on=["user_id", "ts"])
                .join(prev_supp, on=["user_id", "prev_ts", "genre"], how="left")
                .with_columns(pl.col("in_prev").fill_null(0))
                .with_columns((pl.col("p") * pl.col("in_prev")).alias("carried_p"))
                .group_by(["user_id", "ts"])
                .agg(pl.col("carried_p").sum().alias("carried_mass")))

print(f"пар (user, день→предыдущий день): {carried.height:,}")
print("repeat-mass (доля сегодняшней массы на вчерашних жанрах): mean={:.3f} median={:.3f}".format(
    carried["carried_mass"].mean(), carried["carried_mass"].median()))

fig = px.histogram(carried.to_pandas(), x="carried_mass", nbins=50,
                   title="P1 · genre: repeat-mass — доля интереса на ранее посещённых жанрах")
fig.update_layout(xaxis_title="перенесённая масса", yaxis_title="кол-во (user, день)")
fig.show()

пар (user, день→предыдущий день): 255,519
repeat-mass (доля сегодняшней массы на вчерашних жанрах): mean=0.422 median=0.411


## 4. Персональная история vs глобальная популярность (top-10 hit-rate) — решающий тест

repeat-mass (§3) сам по себе неоднозначен: глобальный head тоже покрывает ~42% массы. Поэтому сравним **в лоб**, насколько вчерашний top-10 жанров предсказывает сегодняшний персональный top-10:

- **персональный предиктор** — вчерашний top-10 *этого юзера*;
- **глобальный предиктор** — вчерашний глобальный top-10 (популярность).

Если персональный hit-rate заметно выше глобального — персональная per-(user, item) память несёт сигнал сверх популярности → **идея жива**.

In [6]:
K = 10

def topk(df, by, val, k=K):
    return (df.with_columns(pl.col(val).rank("ordinal", descending=True).over(by).alias("rk"))
              .filter(pl.col("rk") <= k))

# сегодняшний персональный top-K (что предсказываем) + привязка к предыдущему дню юзера
cur_top = topk(cur, ["user_id", "ts"], "p").select(["user_id", "ts", "genre"]) \
            .join(pairs, on=["user_id", "ts"])

# персональный предиктор: вчерашний top-K этого юзера
self_prev = topk(cur, ["user_id", "ts"], "p").select(
    ["user_id", pl.col("ts").alias("prev_ts"), "genre"]).with_columns(pl.lit(1).alias("self_hit"))

# глобальный предиктор: вчерашний глобальный top-K по СУММАРНОЙ массе (популярность по охвату,
# а не по интенсивности у немногих) — это честный, сильный неперсональный baseline
glob = norm.group_by(["ts", "genre"]).agg(pl.col("p").sum().alias("g"))
glob_prev = topk(glob, ["ts"], "g").select(
    [pl.col("ts").alias("prev_ts"), "genre"]).with_columns(pl.lit(1).alias("glob_hit"))

hits = (cur_top
        .join(self_prev, on=["user_id", "prev_ts", "genre"], how="left")
        .join(glob_prev, on=["prev_ts", "genre"], how="left")
        .with_columns([pl.col("self_hit").fill_null(0), pl.col("glob_hit").fill_null(0)])
        .group_by(["user_id", "ts"])
        .agg([pl.col("self_hit").sum().alias("ns"),
              pl.col("glob_hit").sum().alias("ng"),
              pl.len().alias("k")])
        .with_columns([(pl.col("ns") / pl.col("k")).alias("hit_self"),
                       (pl.col("ng") / pl.col("k")).alias("hit_global")]))

hs, hg = hits["hit_self"].mean(), hits["hit_global"].mean()
print(f"top-{K} hit-rate (предсказание сегодняшнего персонального top-{K}):")
print(f"  персональная история : {hs:.3f}")
print(f"  глобальная популярность: {hg:.3f}")
print(f"  отрыв персонального    : +{hs - hg:.3f}  ({(hs/hg - 1)*100:.0f}% относительно)")

cmp = pl.DataFrame({"предиктор": ["персональная история", "глобальная популярность"],
                    "hit_rate": [hs, hg]})
fig = px.bar(cmp.to_pandas(), x="предиктор", y="hit_rate", color="предиктор",
             title=f"P1 · genre: top-{K} hit-rate — персональное vs глобальное")
fig.show()

top-10 hit-rate (предсказание сегодняшнего персонального top-10):
  персональная история : 0.334
  глобальная популярность: 0.354
  отрыв персонального    : +-0.020  (-6% относительно)


### 4b. Тот же тест, но в метрике задачи — NDCG@10

hit-rate@10 — грубая мера (бинарное пересечение множеств, игнорирует ранжирование и веса). Посчитаем **NDCG@10** (метрика TGB, та же что в Table 1) для двух предикторов против истинного сегодняшнего распределения юзера. `persistent` здесь = строка `Persistent Frcst (L)` из статьи; `global popularity` — неперсональный prior. Считаем на выборке (user, день) для скорости.

In [7]:
import numpy as np
from sklearn.metrics import ndcg_score

genres = lab["genre"].unique().to_list()
G = len(genres)
gmap = pl.DataFrame({"genre": genres, "gi": list(range(G))})

samp = pairs.sample(n=3000, seed=1).with_row_index("sid")

def dense(df, valcol):
    arr = np.zeros((samp.height, G), dtype=float)
    arr[df["sid"].to_numpy(), df["gi"].to_numpy()] = df[valcol].to_numpy()
    return arr

yt   = samp.join(cur, on=["user_id", "ts"]).join(gmap, on="genre")                       # истина (сегодня)
pers = samp.join(cur.rename({"ts": "prev_ts"}), on=["user_id", "prev_ts"]).join(gmap, on="genre")  # персональная история
glb  = samp.join(glob.rename({"ts": "prev_ts"}), on=["prev_ts"]).join(gmap, on="genre")  # глобальная популярность

Yt, Pp, Gg = dense(yt, "p"), dense(pers, "p"), dense(glb, "g")

def mean_ndcg(y_true, y_score, k=10):
    keep = np.where(y_true.sum(1) > 0)[0]
    return float(np.mean([ndcg_score(y_true[i:i+1], y_score[i:i+1], k=k) for i in keep]))

nd_pers, nd_glob = mean_ndcg(Yt, Pp), mean_ndcg(Yt, Gg)
print(f"NDCG@10 на выборке {samp.height} (user, день):")
print(f"  персональная история (persistent): {nd_pers:.3f}")
print(f"  глобальная популярность          : {nd_glob:.3f}")
print(f"  отрыв персонального              : {nd_pers - nd_glob:+.3f}")
print(f"\nдля справки (Table 1, genre test): TGNv2 0.469 | MovAvg(M) 0.472 | MovAvg(L) 0.509")

NDCG@10 на выборке 3000 (user, день):
  персональная история (persistent): 0.372
  глобальная популярность          : 0.359
  отрыв персонального              : +0.013

для справки (Table 1, genre test): TGNv2 0.469 | MovAvg(M) 0.472 | MovAvg(L) 0.509


## Вывод по gate P1

In [8]:
print("=== Сводка P1 · tgbn-genre ===\n")
print(f"1. Разреженность: средний носитель {per_uts['support'].mean():.1f}/{N_item} "
      f"→ плотность {per_uts['support'].mean()/N_item*100:.1f}%  "
      f"(плотная матрица почти нулевая, дёшево хранится)")
print(f"2. Глобальный head сильный: top-10 жанров = {pop['cum_share'][9]*100:.0f}% массы")
print(f"3. repeat-mass = {carried['carried_mass'].mean():.3f} "
      f"(но ≈ глобальному head, сам по себе неоднозначен)")
print(f"4. top-10 hit-rate : персональное {hs:.3f} vs глобальное {hg:.3f}  ({hs-hg:+.3f})")
print(f"4b. NDCG@10        : персональное {nd_pers:.3f} vs глобальное {nd_glob:.3f}  ({nd_pers-nd_glob:+.3f})")

alive = nd_pers > nd_glob
print("\n--- ВЕРДИКТ ---")
print(f"Персональный per-(user,item) сигнал ПРЕВОСХОДИТ глобальную популярность в NDCG@10: "
      f"{'ДА' if alive else 'НЕТ'}, но отрыв МОДЕСТНЫЙ (+{nd_pers-nd_glob:.3f}).")
print("→ Идея жива, но genre — консервативный/трудный полигон: улучшения будут малыми,")
print("  мерить надо аккуратно (несколько сидов). Бо́льший headroom ожидается на reddit/token,")
print("  где асимметрия сильнее и в Table 1 TGNv2 заметно ниже MovAvg(L).")
print("\nЗАМЕЧАНИЕ: это 1-дневный persistent-прокси. MovAvg по k>1 дням и обучаемая модель")
print("дадут выше (ср. Table 1). P1 проверяет НАЛИЧИЕ персонального сигнала, не его потолок.")

=== Сводка P1 · tgbn-genre ===

1. Разреженность: средний носитель 10.7/513 → плотность 2.1%  (плотная матрица почти нулевая, дёшево хранится)
2. Глобальный head сильный: top-10 жанров = 42% массы
3. repeat-mass = 0.422 (но ≈ глобальному head, сам по себе неоднозначен)
4. top-10 hit-rate : персональное 0.334 vs глобальное 0.354  (-0.020)
4b. NDCG@10        : персональное 0.372 vs глобальное 0.359  (+0.013)

--- ВЕРДИКТ ---
Персональный per-(user,item) сигнал ПРЕВОСХОДИТ глобальную популярность в NDCG@10: ДА, но отрыв МОДЕСТНЫЙ (+0.013).
→ Идея жива, но genre — консервативный/трудный полигон: улучшения будут малыми,
  мерить надо аккуратно (несколько сидов). Бо́льший headroom ожидается на reddit/token,
  где асимметрия сильнее и в Table 1 TGNv2 заметно ниже MovAvg(L).

ЗАМЕЧАНИЕ: это 1-дневный persistent-прокси. MovAvg по k>1 дням и обучаемая модель
дадут выше (ср. Table 1). P1 проверяет НАЛИЧИЕ персонального сигнала, не его потолок.
